# BASS

**BASS** stands for **Beta, Alpha, Sharpe Ratio, and Standard Deviation** — four foundational metrics used to evaluate the performance and risk characteristics of a stock or portfolio.

Together, these measures help investors understand:

- **Beta** – How sensitive a stock is to overall market movements (systematic risk).
- **Alpha** – The excess return generated relative to a benchmark after adjusting for risk.
- **Sharpe Ratio** – The return earned per unit of risk taken.
- **Standard Deviation** – The volatility of returns, measuring total risk.

In this notebook, we will compute and interpret each of these metrics using real (or simulated) market data, and explore how they provide a structured framework for performance evaluation.

In [1]:
# Uncomment the line below to install the yfinance package if it's not already installed
# !pip install yfinance

Import relevant libraries

In [ ]:
import pandas as pd
import yfinance as yf

We will retrieve a set of stocks from YahooFinance, alongside an index for reference, over a certain time period

In [3]:
all_stocks = [
  "C52.SI", "T39.SI", "S68.SI", "G13.SI", "V03.SI" , "U11.SI", "C07.SI", "D05.SI",
  "Z74.SI", "D01.SI", "O39.SI", "S63.SI", "A17U.SI", "BN4.SI","BS6.SI", "M44U.SI",
  "H78.SI", "Y92.SI", "C38U.SI", "U14.SI", "N2IU.SI", "F34.SI", "C09.SI", "J36.SI",
  "S58.SI", "C6L.SI", "U96.SI", "1810.HK", "9999.HK", "7500.HK", "9618.HK", "1024.HK",
  "3690.HK", "6618.HK"
]
benchmark_symbol = "^STI"
start = "2022-09-01"
end = "2023-09-06"

With that, let us define the functions to compute BASS. If you are interested to learn more about what each metric does, please refer to the explanation below. Else you can simply skip to the coding portion.

---

**BASS Formulas**

Given daily returns:
- $r_s$ = stock daily returns  
- $r_b$ = benchmark daily returns  
- $T$ = number of trading days  
- 252 = number of trading days per year  


### 1. Beta (β)

Measures sensitivity of the stock to the benchmark:

$$
\beta = \frac{\operatorname{Cov}(r_s, r_b)}{\operatorname{Var}(r_b)}
$$


### 2. Alpha (α)

Annualised excess return after adjusting for market exposure:

$$
\alpha = \left( \bar{r}_s \cdot 252 - \beta \cdot \bar{r}_b \cdot 252 \right) \times 100
$$

Where:

- $\bar{r}_s$ = mean daily stock return  
- $\bar{r}_b$ = mean daily benchmark return  

Alpha is expressed in percentage terms.


### 3. Standard Deviation (σ)

Measures volatility of daily stock returns:

$$
\sigma = \operatorname{Std}(r_s) \times 100
$$

Expressed as a percentage (daily volatility).


### 4. Sharpe Ratio (SR)

Risk-adjusted return (assuming risk-free rate ≈ 0):

Daily Sharpe Ratio:

$$
SR_{daily} = \frac{\bar{r}_s}{\operatorname{Std}(r_s)}
$$

Annualised Sharpe Ratio:

$$
SR_{annual} = SR_{daily} \times \sqrt{252}
$$


In [4]:
def BASS(stock_symbol, benchmark_symbol, start, end):
  df = pd.DataFrame()
  df['benchmark'] = yf.Ticker(benchmark_symbol).history(start=start, end=end).Close
  df['stock'] = yf.Ticker(stock_symbol).history(start=start, end=end).Close
  df['stock_returns'] = df['stock'].pct_change()
  df['benchmark_returns'] = df['benchmark'].pct_change()

  df = df.dropna()
  if len(df) == 0:
    raise Exception(f'ERROR:yfinance:{stock_symbol}: No timezone found, symbol may be delisted')

  # Beta calculations
  beta = round(df['benchmark_returns'].cov(df['stock_returns']) / df['benchmark_returns'].var(), 2)

  # Alpha calculations
  benchmark_yearly_returns = df['benchmark_returns'].mean() * 252
  stock_yearly_returns = df['stock_returns'].mean() * 252
  alpha = round((stock_yearly_returns - beta * benchmark_yearly_returns) * 100, 2)

  # Standard Deviation
  std_dev = round(df['stock_returns'].std() * 100, 2)

  # Sharpe Ratio of stock
  avg_returns = df['stock_returns'].mean()
  std = df['stock_returns'].std()
  daily_SR = avg_returns / std
  annual_SR = round(daily_SR * (252**0.5), 2)
  
  return beta, alpha, std_dev, annual_SR

In [5]:
# Calculate the BASS for each stock
bass_dictionary = {
  "stock": [],
  "beta": [],
  "alpha": [],
  "std_dev": [],
  "annual_SR": []
}

for stock in all_stocks:
  try:
    beta, alpha, std_dev, annual_SR = BASS(stock, benchmark_symbol, start, end)
    bass_dictionary['stock'].append(stock)
    bass_dictionary['beta'].append(beta)
    bass_dictionary['alpha'].append(alpha)
    bass_dictionary['std_dev'].append(std_dev)
    bass_dictionary['annual_SR'].append(annual_SR)
  except Exception as e:
    print(e)

$T39.SI: possibly delisted; no timezone found
/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_28979/2452925945.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['stock_returns'] = df['stock'].pct_change()


ERROR:yfinance:T39.SI: No timezone found, symbol may be delisted


/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_28979/2452925945.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['stock_returns'] = df['stock'].pct_change()
/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_28979/2452925945.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  df['stock_returns'] = df['stock'].pct_change()
/var/folders/5v/9xrz7nmd29s2z029pvbqs8r00000gn/T/ipykernel_28979/2452925945.py:5: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to

In [6]:
# Create a dataframe to house all the BASS data for the various stocks in STI Index
bass = pd.DataFrame(bass_dictionary, columns = bass_dictionary.keys())

# Sort the data by order of Sharpe Ratio
bass = bass.sort_values(by='annual_SR', ascending=False).reset_index(drop=True)
bass

,stock,beta,alpha,std_dev,annual_SR
0,C6L.SI,0.82,33.00,1.09,1.94
1,U96.SI,0.86,50.50,1.77,1.82
2,BS6.SI,0.64,66.53,2.36,1.79
3,BN4.SI,1.01,42.40,1.53,1.77
4,Y92.SI,1.53,200.44,11.15,1.14
5,G13.SI,0.69,18.69,1.19,1.01
6,O39.SI,0.94,10.20,0.75,0.90
7,U11.SI,1.06,10.84,0.95,0.76
8,D05.SI,1.13,9.40,0.95,0.67
9,9999.HK,1.61,30.81,3.02,0.54
